<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets nltk -q

import random
import numpy as np
import torch
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")

Setup done!


In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print(dataset)
print("\n--- Sample record ---")
print(dataset["train"][1]["text"])
print(dataset["train"][2]["text"])

# Extract non-empty lines
train_text = [ex["text"].strip() for ex in dataset["train"] if ex["text"].strip()]
val_text   = [ex["text"].strip() for ex in dataset["validation"] if ex["text"].strip()]
test_text  = [ex["text"].strip() for ex in dataset["test"] if ex["text"].strip()]

print(f"\nTrain lines: {len(train_text)}")
print(f"Val lines:   {len(val_text)}")
print(f"Test lines:  {len(test_text)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

--- Sample record ---
 = Valkyria Chronicles III = 



Train lines: 23767
Val lines:   2461
Test lines:  2891


In [ ]:
from collections import Counter, defaultdict
import numpy as np

# Tokenize
def tokenize(lines):
    tokens = []
    for line in lines:
        tokens.extend(line.lower().split())
    return tokens

train_tokens = tokenize(train_text)
val_tokens   = tokenize(val_text)
test_tokens  = tokenize(test_text)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Val tokens:   {len(val_tokens):,}")
print(f"Test tokens:  {len(test_tokens):,}")

# Build N-gram model (trigram with Laplace smoothing)
class NgramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()

    def train(self, tokens):
        self.vocab = set(tokens)
        self.vocab_size = len(self.vocab)
        for i in range(len(tokens) - self.n + 1):
            context = tuple(tokens[i:i+self.n-1])
            word = tokens[i+self.n-1]
            self.ngram_counts[context][word] += 1
            self.context_counts[context] += 1
        print(f"N-gram model trained! Vocab size: {self.vocab_size:,}")

    def prob(self, context, word):
        context = tuple(context)
        # Laplace smoothing
        count = self.ngram_counts[context][word] + 1
        total = self.context_counts[context] + self.vocab_size
        return count / total

    def perplexity(self, tokens):
        log_prob = 0
        n_words = 0
        for i in range(self.n-1, len(tokens)):
            context = tuple(tokens[i-self.n+1:i])
            word = tokens[i]
            log_prob += np.log(self.prob(context, word))
            n_words += 1
        return np.exp(-log_prob / n_words)

    def generate(self, seed_words, num_words=20):
        result = list(seed_words)
        for _ in range(num_words):
            context = tuple(result[-(self.n-1):])
            if context in self.ngram_counts:
                candidates = list(self.ngram_counts[context].keys())
                probs = [self.prob(context, w) for w in candidates]
                probs = np.array(probs) / sum(probs)
                word = np.random.choice(candidates, p=probs)
            else:
                word = np.random.choice(list(self.vocab))
            result.append(word)
        return ' '.join(result)

# Train trigram model
trigram = NgramModel(n=3)
trigram.train(train_tokens)

Train tokens: 2,051,910
Val tokens:   213,886
Test tokens:  241,211
N-gram model trained! Vocab size: 66,649


In [ ]:
# Evaluate perplexity
print("Computing perplexity (this may take a minute)...")
val_perplexity = trigram.perplexity(val_tokens[:50000])
test_perplexity = trigram.perplexity(test_tokens[:50000])

print(f"\n--- Trigram Model Results ---")
print(f"Validation Perplexity: {val_perplexity:.2f}")
print(f"Test Perplexity:       {test_perplexity:.2f}")

# Generate text samples
print("\n--- Generated Text Samples ---")
seeds = [
    ["the", "history"],
    ["in", "the"],
    ["science", "and"]
]

for seed in seeds:
    print(f"\nSeed: '{' '.join(seed)}'")
    print("Output:", trigram.generate(seed, num_words=20))

Computing perplexity (this may take a minute)...

--- Trigram Model Results ---
Validation Perplexity: 30056.27
Test Perplexity:       31898.52

--- Generated Text Samples ---

Seed: 'the history'
Output: the history of jain philosophy in maynooth college , at 04 : 10 = = = = = = nebraska highway 88

Seed: 'in the'
Output: in the program , the establishment of an eighth in the united states . the drug @-@ discovery . the faster speed

Seed: 'science and'
Output: science and math are viewed as definitive . woody guthrie , robert fassl ( w. earl brown ) murders and partly governs


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Build vocabulary
def build_vocab(tokens, max_vocab=10000):
    counter = Counter(tokens)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_tokens)
idx2word = {v: k for k, v in vocab.items()}
print(f"Vocabulary size: {len(vocab)}")

# Encode tokens
def encode_tokens(tokens, vocab):
    return [vocab.get(t, 1) for t in tokens]

train_ids = encode_tokens(train_tokens, vocab)
val_ids   = encode_tokens(val_tokens, vocab)
test_ids  = encode_tokens(test_tokens, vocab)

# Dataset
class LMDataset(Dataset):
    def __init__(self, token_ids, seq_len=64):
        self.data = token_ids
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_dataset = LMDataset(train_ids)
val_dataset   = LMDataset(val_ids)
test_dataset  = LMDataset(test_ids)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

print("Datasets ready!")
print(f"Train batches: {len(train_loader)}")

Vocabulary size: 10000
Datasets ready!
Train batches: 32061


In [ ]:
import torch
import torch.nn as nn
import numpy as np

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, _ = self.lstm(embedded)
        return self.fc(self.dropout(output))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Subset + larger batch for speed
train_dataset_small = LMDataset(train_ids[:100000], seq_len=64)
train_loader_small  = DataLoader(train_dataset_small, batch_size=256, shuffle=True)

lstm_lm = LSTMLanguageModel(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(lstm_lm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_lm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
            total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    return avg_loss, np.exp(avg_loss)

EPOCHS = 5
print("\nTraining LSTM Language Model...")
for epoch in range(EPOCHS):
    train_loss = train_epoch(lstm_lm, train_loader_small, optimizer, criterion, device)
    val_loss, val_ppl = evaluate_lm(lstm_lm, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")

print("\nDone!")

Using device: cuda

Training LSTM Language Model...
Epoch 1/5 | Train Loss: 5.9247 | Val Loss: 6.0832 | Val PPL: 438.44
Epoch 2/5 | Train Loss: 4.9794 | Val Loss: 6.0771 | Val PPL: 435.75
Epoch 3/5 | Train Loss: 4.5026 | Val Loss: 6.1698 | Val PPL: 478.10
Epoch 4/5 | Train Loss: 4.1493 | Val Loss: 6.2955 | Val PPL: 542.13
Epoch 5/5 | Train Loss: 3.8061 | Val Loss: 6.4581 | Val PPL: 637.86

Done!


In [ ]:
# N-gram perplexity
print("Computing N-gram perplexity...")
val_ppl_ngram  = trigram.perplexity(val_tokens[:10000])
test_ppl_ngram = trigram.perplexity(test_tokens[:10000])

# LSTM test perplexity
test_loss, test_ppl_lstm = evaluate_lm(lstm_lm, test_loader, criterion, device)

print("\n" + "="*55)
print(f"{'Model':<20} {'Val PPL':>15} {'Test PPL':>15}")
print("="*55)
print(f"{'Trigram (N-gram)':<20} {val_ppl_ngram:>15.2f} {test_ppl_ngram:>15.2f}")
print(f"{'LSTM':<20} {637.86:>15.2f} {test_ppl_lstm:>15.2f}")
print("="*55)

# Text generation
def lstm_generate(model, vocab, idx2word, seed_words, num_words=20, device='cpu'):
    model.eval()
    tokens = [vocab.get(w, 1) for w in seed_words]
    with torch.no_grad():
        for _ in range(num_words):
            x = torch.tensor([tokens[-64:]], dtype=torch.long).to(device)
            output = model(x)
            probs = torch.softmax(output[0, -1], dim=0).cpu().numpy()
            next_id = np.random.choice(len(probs), p=probs)
            tokens.append(next_id)
    return ' '.join([idx2word.get(t, '<UNK>') for t in tokens])

print("\n--- Text Generation Samples ---")
seeds = [["the", "history"], ["in", "the"], ["science", "and"]]
for seed in seeds:
    print(f"\nSeed: '{' '.join(seed)}'")
    print("Trigram:", trigram.generate(seed, num_words=20))
    print("LSTM   :", lstm_generate(lstm_lm, vocab, idx2word, seed, num_words=20, device=device))

Computing N-gram perplexity...

Model                        Val PPL        Test PPL
Trigram (N-gram)            30845.02        29547.91
LSTM                          637.86          561.84

--- Text Generation Samples ---

Seed: 'the history'
Trigram: the history of the dynasty , maturing to a jumping board and an increased population brought hunters who stirred the curiosity of
LSTM   : the history @-@ face of <UNK> , <UNK> and having non @-@ <UNK> for over their assistant country . = = a

Seed: 'in the'
Trigram: in the united states , focus on humor rather than building a hvdc transmission line between believability and pushing something as innocent
LSTM   : in the absence of the <UNK> , is expressed from <UNK> a different species . the scenic head of a main thread

Seed: 'science and'
Trigram: science and culture = = = = the plunketts creek is named in 1884 , the altar of forgiveness and the hss
LSTM   : science and game has no longer was skin as the <UNK> was inspired <UNK> for t